<a href="https://colab.research.google.com/github/harinijk/DeepFake/blob/main/AIorReal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers timm kaggle -q

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d xhlulu/140k-real-and-fake-faces

!unzip -q 140k-real-and-fake-faces.zip

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from torch.utils.data import Subset
import random

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

from transformers import pipeline

from PIL import Image

In [ ]:
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using:", DEVICE)


In [ ]:
random.seed(42)

In [ ]:
train_transform = transforms.Compose([

    transforms.Resize((128,128)),

    transforms.RandomHorizontalFlip(),

    transforms.RandomRotation(10),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


test_transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
train_indices = random.sample(
    range(len(train_dataset)),
    8000
)

valid_indices = random.sample(
    range(len(valid_dataset)),
    2000
)

test_indices = random.sample(
    range(len(test_dataset)),
    2000
)

In [ ]:
train_dataset = datasets.ImageFolder(
    "real_vs_fake/real-vs-fake/train",
    transform=train_transform
)

valid_dataset = datasets.ImageFolder(
    "real_vs_fake/real-vs-fake/valid",
    transform=test_transform
)

test_dataset = datasets.ImageFolder(
    "real_vs_fake/real-vs-fake/test",
    transform=test_transform
)


In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

print(train_dataset.classes)

In [ ]:
images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2,5, figsize=(12,5))

for i, ax in enumerate(axes.flatten()):

    img = images[i].permute(1,2,0).numpy()

    img = img * np.array([0.229,0.224,0.225])
    img = img + np.array([0.485,0.456,0.406])

    img = np.clip(img, 0, 1)

    ax.imshow(img)

    ax.set_title(
        train_dataset.classes[labels[i]]
    )

    ax.axis("off")

plt.show()

In [ ]:
model = models.resnet18(
    pretrained=True
)

for param in model.parameters():
    param.requires_grad = False


model.fc = nn.Sequential(

    nn.Linear(
        model.fc.in_features,
        512
    ),

    nn.ReLU(),

    nn.Dropout(0.4),

    nn.Linear(512, 2)
)

model = model.to(DEVICE)


In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.fc.parameters(),
    lr=0.0005
)

scheduler = optim.lr_scheduler.StepLR(
    optimizer,
    step_size=2,
    gamma=0.5
)

In [ ]:
EPOCHS = 5

train_losses = []
valid_losses = []

best_valid_loss = float("inf")


for epoch in range(EPOCHS):

    model.train()

    train_loss = 0

    for images, labels in train_loader:

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        train_loss += loss.item()


   #validation

    model.eval()

    valid_loss = 0

    with torch.no_grad():

        for images, labels in valid_loader:

            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            valid_loss += loss.item()


    train_loss /= len(train_loader)
    valid_loss /= len(valid_loader)

    train_losses.append(train_loss)
    valid_losses.append(valid_loss)

    scheduler.step()

    print(
        f"Epoch {epoch+1}"
        f" | Train Loss: {train_loss:.4f}"
        f" | Valid Loss: {valid_loss:.4f}"
    )

    if valid_loss < best_valid_loss:

        best_valid_loss = valid_loss

        torch.save(
            model.state_dict(),
            "best_model.pth"
        )

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(train_losses, label="Train")
plt.plot(valid_losses, label="Validation")

plt.title("Training vs Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.legend()

plt.show()

In [ ]:
model.load_state_dict(
    torch.load("best_model.pth")
)

In [ ]:
model.eval()

predictions = []
true_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(DEVICE)

        outputs = model(images)

        _, preds = torch.max(outputs, 1)

        predictions.extend(
            preds.cpu().numpy()
        )

        true_labels.extend(
            labels.numpy()
        )


In [ ]:
accuracy = accuracy_score(
    true_labels,
    predictions
)

precision = precision_score(
    true_labels,
    predictions
)

recall = recall_score(
    true_labels,
    predictions
)

f1 = f1_score(
    true_labels,
    predictions
)

print("\nCNN RESULTS")
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)

In [ ]:
print(
    classification_report(
        true_labels,
        predictions,
        target_names=["Fake", "Real"]
    )
)
cm = confusion_matrix(
    true_labels,
    predictions
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Fake", "Real"]
)

disp.plot()

plt.show()

In [ ]:
fig, axes = plt.subplots(2,5, figsize=(14,6))

sample_images, sample_labels = next(
    iter(test_loader)
)

sample_images = sample_images.to(DEVICE)

outputs = model(sample_images)

_, preds = torch.max(outputs, 1)

sample_images = sample_images.cpu()

for i, ax in enumerate(axes.flatten()):

    img = sample_images[i].permute(1,2,0).numpy()

    img = img * np.array([0.229,0.224,0.225])
    img = img + np.array([0.485,0.456,0.406])

    img = np.clip(img, 0, 1)

    ax.imshow(img)

    pred_label = train_dataset.classes[
        preds[i]
    ]

    true_label = train_dataset.classes[
        sample_labels[i]
    ]

    ax.set_title(
        f"P:{pred_label}\nT:{true_label}"
    )

    ax.axis("off")

plt.tight_layout()

plt.show()

In [ ]:
vlm = pipeline(
    "image-classification",
    model="google/vit-base-patch16-224"
)

example_path = (
    "real_vs_fake/real-vs-fake/test/fake"
)

example_image = os.listdir(
    example_path
)[0]

full_path = os.path.join(
    example_path,
    example_image
)

image = Image.open(full_path)

results = vlm(image)

print(results)